In [ ]:
import pygame
import sys
import heapq
import time
from collections import deque
import pandas as pd

pygame.init()

# Window setup
WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Towers of Hanoi - AI Mode")

# Fonts
FONT = pygame.font.SysFont("Arial", 20)
BIG_FONT = pygame.font.SysFont("Arial", 48, bold=True)

# Colors
WHITE = (250, 250, 250)
BLACK = (0, 0, 0)
GRAY = (180, 180, 180)
RED = (200, 50, 50)
BLUE = (50, 100, 200)
GREEN = (50, 200, 100)
BG_COLOR = (240, 245, 255)

# Load sounds with error handling
try:
    move_sound = pygame.mixer.Sound("move.mp3")
except pygame.error as e:
    print(f"Warning: Could not load move.mp3 - {e}")
    move_sound = None
try:
    victory_sound = pygame.mixer.Sound("victory.mp3")
except pygame.error as e:
    print(f"Warning: Could not load victory.mp3 - {e}")
    victory_sound = None

# Towers positions
tower_x = [250, 450, 650]

# Draw towers and disks
def draw_towers(state, num_disks, shake=False):
    screen.fill(BG_COLOR)
    offset = 3 if shake else 0
    pygame.draw.rect(screen, BLACK, (150 - offset, 500, 600, 10))
    for x in tower_x:
        pygame.draw.rect(screen, BLACK, (x - 10 + offset, 200, 20, 300))
    colors = [pygame.Color("#FFADAD"), pygame.Color("#FFD6A5"), pygame.Color("#FDFFB6"),
              pygame.Color("#CAFFBF"), pygame.Color("#9BF6FF"), pygame.Color("#A0C4FF")]
    for i in range(3):
        tower = state[i]
        for j, disk in enumerate(tower):
            width = 30 + disk * 20
            rect = pygame.Rect(tower_x[i] - width // 2 + offset, 500 - (j + 1) * 20, width, 20)
            pygame.draw.rect(screen, colors[disk - 1], rect)
            pygame.draw.rect(screen, BLACK, rect, 2)

# Helper function to convert state to tuple
def state_to_tuple(state):
    return tuple(tuple(peg) for peg in state)

# Heuristic for A*
def heuristic(state, goal_state):
    return sum(1 for i in range(3) for disk in state[i] if disk not in goal_state[i])

# A* algorithm for Hanoi
def a_star_hanoi(start_state, goal_state, timeout=5.0, max_iterations=10000):
    start_time = time.perf_counter()
    visited = set()
    heap = [(0, 0, start_state, [])]  # (f_score, g_score, state, path)
    states_explored = 0
    iterations = 0

    while heap and (time.perf_counter() - start_time) < timeout and iterations < max_iterations:
        pygame.event.pump()
        iterations += 1
        _, g_score, state, path = heapq.heappop(heap)
        state_tuple = state_to_tuple(state)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)
        states_explored += 1
        if state == goal_state:
            return path, time.perf_counter() - start_time, states_explored
        for from_peg in range(3):
            if not state[from_peg]:
                continue
            disk = state[from_peg][-1]
            for to_peg in range(3):
                if from_peg == to_peg:
                    continue
                if not state[to_peg] or state[to_peg][-1] > disk:
                    new_state = [list(peg) for peg in state]
                    new_state[from_peg].pop()
                    new_state[to_peg].append(disk)
                    new_state_tuple = state_to_tuple(new_state)
                    if new_state_tuple not in visited:
                        heapq.heappush(heap, (
                            g_score + 1 + heuristic(new_state, goal_state),
                            g_score + 1,
                            new_state,
                            path + [(from_peg, to_peg)]
                        ))
    return [], time.perf_counter() - start_time, states_explored

# BFS algorithm for Hanoi
def bfs_hanoi(start_state, goal_state, timeout=5.0, max_iterations=10000):
    start_time = time.perf_counter()
    queue = deque([(start_state, [])])
    visited = set()
    states_explored = 0
    iterations = 0

    while queue and (time.perf_counter() - start_time) < timeout and iterations < max_iterations:
        pygame.event.pump()
        iterations += 1
        state, path = queue.popleft()
        state_tuple = state_to_tuple(state)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)
        states_explored += 1
        if state == goal_state:
            return path, time.perf_counter() - start_time, states_explored
        for from_peg in range(3):
            if not state[from_peg]:
                continue
            disk = state[from_peg][-1]
            for to_peg in range(3):
                if from_peg == to_peg:
                    continue
                if not state[to_peg] or state[to_peg][-1] > disk:
                    new_state = [list(peg) for peg in state]
                    new_state[from_peg].pop()
                    new_state[to_peg].append(disk)
                    new_state_tuple = state_to_tuple(new_state)
                    if new_state_tuple not in visited:
                        queue.append((new_state, path + [(from_peg, to_peg)]))
    return [], time.perf_counter() - start_time, states_explored

# DFS algorithm for Hanoi
def dfs_hanoi(start_state, goal_state, depth_limit=50, timeout=5.0, max_iterations=10000):
    start_time = time.perf_counter()
    stack = [(start_state, [], 0)]  # (state, path, depth)
    visited = set()
    states_explored = 0
    iterations = 0

    while stack and (time.perf_counter() - start_time) < timeout and iterations < max_iterations:
        pygame.event.pump()
        iterations += 1
        state, path, depth = stack.pop()
        state_tuple = state_to_tuple(state)
        if state_tuple in visited or depth > depth_limit:
            continue
        visited.add(state_tuple)
        states_explored += 1
        if state == goal_state:
            return path, time.perf_counter() - start_time, states_explored
        for from_peg in range(3):
            if not state[from_peg]:
                continue
            disk = state[from_peg][-1]
            for to_peg in range(3):
                if from_peg == to_peg:
                    continue
                if not state[to_peg] or state[to_peg][-1] > disk:
                    new_state = [list(peg) for peg in state]
                    new_state[from_peg].pop()
                    new_state[to_peg].append(disk)
                    new_state_tuple = state_to_tuple(new_state)
                    if new_state_tuple not in visited:
                        stack.append((new_state, path + [(from_peg, to_peg)], depth + 1))
    return [], time.perf_counter() - start_time, states_explored

# UCS algorithm for Hanoi
def ucs_hanoi(start_state, goal_state, timeout=5.0, max_iterations=10000):
    start_time = time.perf_counter()
    heap = [(0, start_state, [])]  # (cost, state, path)
    visited = set()
    states_explored = 0
    iterations = 0

    while heap and (time.perf_counter() - start_time) < timeout and iterations < max_iterations:
        pygame.event.pump()
        iterations += 1
        cost, state, path = heapq.heappop(heap)
        state_tuple = state_to_tuple(state)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)
        states_explored += 1
        if state == goal_state:
            return path, time.perf_counter() - start_time, states_explored
        for from_peg in range(3):
            if not state[from_peg]:
                continue
            disk = state[from_peg][-1]
            for to_peg in range(3):
                if from_peg == to_peg:
                    continue
                if not state[to_peg] or state[to_peg][-1] > disk:
                    new_state = [list(peg) for peg in state]
                    new_state[from_peg].pop()
                    new_state[to_peg].append(disk)
                    new_state_tuple = state_to_tuple(new_state)
                    if new_state_tuple not in visited:
                        heapq.heappush(heap, (cost + 1, new_state, path + [(from_peg, to_peg)]))
    return [], time.perf_counter() - start_time, states_explored

# Auto solve using specified algorithm
def auto_solve(state, num_disks, algorithm="A*"):
    goal_state = [[], [], list(reversed(range(1, num_disks + 1)))]
    if algorithm == "A*":
        moves, exec_time, states_explored = a_star_hanoi(state, goal_state)
    elif algorithm == "BFS":
        moves, exec_time, states_explored = bfs_hanoi(state, goal_state)
    elif algorithm == "DFS":
        moves, exec_time, states_explored = dfs_hanoi(state, goal_state)
    elif algorithm == "UCS":
        moves, exec_time, states_explored = ucs_hanoi(state, goal_state)
    else:
        return 0, 0, 0
    for move in moves:
        from_peg, to_peg = move
        if not state[from_peg]:
            continue
        disk = state[from_peg].pop()
        state[to_peg].append(disk)
        draw_towers(state, num_disks)
        pygame.display.flip()
        if move_sound:
            move_sound.play()
        pygame.time.delay(400)
    return len(moves), exec_time, states_explored

# Welcome screen
def welcome_screen():
    selected_disks = 3
    start_button = pygame.Rect(350, 400, 200, 50)
    minus_button = pygame.Rect(350, 250, 40, 40)
    plus_button = pygame.Rect(510, 250, 40, 40)
    while True:
        screen.fill(BG_COLOR)
        title = BIG_FONT.render("Towers of Hanoi", True, BLUE)
        screen.blit(title, (WIDTH // 2 - title.get_width() // 2, 100))
        disk_text = FONT.render(f"Number of disks: {selected_disks}", True, BLACK)
        screen.blit(disk_text, (WIDTH // 2 - disk_text.get_width() // 2, 200))
        pygame.draw.rect(screen, RED, minus_button)
        pygame.draw.rect(screen, GREEN, plus_button)
        screen.blit(FONT.render("-", True, WHITE), (minus_button.x + 12, minus_button.y + 5))
        screen.blit(FONT.render("+", True, WHITE), (plus_button.x + 12, plus_button.y + 5))
        pygame.draw.rect(screen, BLUE, start_button)
        screen.blit(FONT.render("Start Game", True, WHITE), (start_button.x + 30, start_button.y + 10))
        pygame.display.flip()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == pygame.MOUSEBUTTONDOWN:
                if minus_button.collidepoint(event.pos) and selected_disks > 3:
                    selected_disks -= 1
                elif plus_button.collidepoint(event.pos) and selected_disks < 6:
                    selected_disks += 1
                elif start_button.collidepoint(event.pos):
                    return selected_disks

# Win screen
def game_won():
    if victory_sound:
        victory_sound.play()
    for _ in range(4):
        draw_towers([[], [], []], 0, shake=True)
        pygame.display.flip()
        pygame.time.delay(100)
        draw_towers([[], [], []], 0, shake=False)
        pygame.display.flip()
        pygame.time.delay(100)
    screen.fill(BG_COLOR)
    msg = BIG_FONT.render("Well Done! Puzzle Solved!", True, GREEN)
    screen.blit(msg, (WIDTH // 2 - msg.get_width() // 2, HEIGHT // 2 - 50))
    pygame.display.flip()
    pygame.time.delay(2000)

# Compare algorithms
def compare_algorithms(num_disks, towers):
    start_state = [list(reversed(range(1, num_disks + 1))), [], []]
    goal_state = [[], [], list(reversed(range(1, num_disks + 1)))]
    results = []
    original_state = [list(peg) for peg in towers]
    for algo in ["A*", "BFS", "DFS", "UCS"]:
        pygame.event.pump()
        state = [list(peg) for peg in start_state]
        try:
            moves, exec_time, states_explored = auto_solve(state, num_disks, algo)
            results.append({
                "Algorithm": algo,
                "Moves": moves,
                "Time (s)": round(exec_time, 6),
                "States Explored": states_explored
            })
        except Exception as e:
            print(f"Error in {algo}: {e}")
            results.append({
                "Algorithm": algo,
                "Moves": "Failed",
                "Time (s)": 0.0,
                "States Explored": 0
            })
        draw_towers(original_state, num_disks)
        pygame.display.flip()
        pygame.time.delay(500)
    screen.fill(BG_COLOR)
    y_offset = 50
    title = BIG_FONT.render("Algorithm Comparison", True, BLUE)
    screen.blit(title, (WIDTH // 2 - title.get_width() // 2, y_offset))
    y_offset += 60
    headers = ["Algorithm", "Moves", "Time (s)", "States Explored"]
    col_widths = [100, 80, 120, 120]
    x_start = 250
    header_bg = pygame.Surface((sum(col_widths), 30))
    header_bg.fill(GRAY)
    screen.blit(header_bg, (x_start, y_offset))
    for i, header in enumerate(headers):
        text = FONT.render(header, True, BLACK)
        screen.blit(text, (x_start + sum(col_widths[:i]) + 10, y_offset + 5))
    y_offset += 30
    for result in results:
        row_bg = pygame.Surface((sum(col_widths), 25))
        row_bg.fill(WHITE)
        screen.blit(row_bg, (x_start, y_offset))
        row = [result["Algorithm"], str(result["Moves"]), f"{result['Time (s)']:.6f}", str(result["States Explored"])]
        for i, value in enumerate(row):
            text = FONT.render(value, True, BLACK)
            screen.blit(text, (x_start + sum(col_widths[:i]) + 10, y_offset + 5))
        y_offset += 25
    pygame.display.flip()
    pygame.time.delay(3000)
    draw_towers(original_state, num_disks)
    pygame.display.flip()
    df = pd.DataFrame(results)
    print("\nPerformance Comparison:")
    print(df)
    df.to_csv("hanoi_performance.csv", index=False)
    return original_state

# Main game loop
def main():
    num_disks = welcome_screen()
    towers = [list(reversed(range(1, num_disks + 1))), [], []]
    a_star_button = pygame.Rect(680, 40, 180, 40)
    bfs_button = pygame.Rect(680, 90, 180, 40)
    dfs_button = pygame.Rect(680, 140, 180, 40)
    ucs_button = pygame.Rect(680, 190, 180, 40)
    restart_button = pygame.Rect(680, 240, 180, 40)
    compare_button = pygame.Rect(680, 290, 180, 40)
    dragging = False
    selected_peg = None
    while True:
        draw_towers(towers, num_disks)
        pygame.draw.rect(screen, RED, a_star_button)
        screen.blit(FONT.render("Solve with A*", True, WHITE), (a_star_button.x + 20, a_star_button.y + 8))
        pygame.draw.rect(screen, RED, bfs_button)
        screen.blit(FONT.render("Solve with BFS", True, WHITE), (bfs_button.x + 20, bfs_button.y + 8))
        pygame.draw.rect(screen, RED, dfs_button)
        screen.blit(FONT.render("Solve with DFS", True, WHITE), (dfs_button.x + 20, dfs_button.y + 8))
        pygame.draw.rect(screen, RED, ucs_button)
        screen.blit(FONT.render("Solve with UCS", True, WHITE), (ucs_button.x + 20, ucs_button.y + 8))
        pygame.draw.rect(screen, BLUE, restart_button)
        screen.blit(FONT.render("Restart", True, WHITE), (restart_button.x + 50, restart_button.y + 8))
        pygame.draw.rect(screen, GREEN, compare_button)
        screen.blit(FONT.render("Compare Algos", True, WHITE), (compare_button.x + 20, compare_button.y + 8))
        pygame.display.flip()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == pygame.MOUSEBUTTONDOWN:
                if a_star_button.collidepoint(event.pos):
                    auto_solve(towers, num_disks, "A*")
                    if towers == [[], [], list(reversed(range(1, num_disks + 1)))]:
                        game_won()
                        return main()
                elif bfs_button.collidepoint(event.pos):
                    auto_solve(towers, num_disks, "BFS")
                    if towers == [[], [], list(reversed(range(1, num_disks + 1)))]:
                        game_won()
                        return main()
                elif dfs_button.collidepoint(event.pos):
                    auto_solve(towers, num_disks, "DFS")
                    if towers == [[], [], list(reversed(range(1, num_disks + 1)))]:
                        game_won()
                        return main()
                elif ucs_button.collidepoint(event.pos):
                    auto_solve(towers, num_disks, "UCS")
                    if towers == [[], [], list(reversed(range(1, num_disks + 1)))]:
                        game_won()
                        return main()
                elif restart_button.collidepoint(event.pos):
                    return main()
                elif compare_button.collidepoint(event.pos):
                    towers = compare_algorithms(num_disks, towers)
                else:
                    x = event.pos[0]
                    for i, tx in enumerate(tower_x):
                        if abs(x - tx) < 50:
                            if towers[i]:
                                dragging = True
                                selected_peg = i
            elif event.type == pygame.MOUSEBUTTONUP and dragging:
                x = event.pos[0]
                for i, tx in enumerate(tower_x):
                    if abs(x - tx) < 50:
                        if i != selected_peg and (not towers[i] or towers[i][-1] > towers[selected_peg][-1]):
                            towers[i].append(towers[selected_peg].pop())
                            if move_sound:
                                move_sound.play()
                            if towers == [[], [], list(reversed(range(1, num_disks + 1)))]:
                                game_won()
                                return main()
                        break
                dragging = False
                selected_peg = None

main()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html

Performance Comparison:
  Algorithm  Moves  Time (s)  States Explored
0        A*      7  0.000377               18
1       BFS      7  0.001237               25
2       DFS     13  0.000624               14
3       UCS      7  0.000518               20

Performance Comparison:
  Algorithm  Moves  Time (s)  States Explored
0        A*      7  0.000306               18
1       BFS      7  0.001530               25
2       DFS     13  0.000252               14
3       UCS      7  0.000278               20

Performance Comparison:
  Algorithm  Moves  Time (s)  States Explored
0        A*     15  0.000793               54
1       BFS     15  0.004046               71
2       DFS     40  0.003714               41
3       UCS     15  0.004205               66
